## 基礎 CNN 與 ResNet18 晶圓圖缺陷分類

- 目標：精通使用 PyTorch 建立適用於二維晶圓圖（Wafer Bin Map）缺陷分類的卷積神經網路（CNN）架構。掌握如何微調（Fine-tune）業界標準模型 ResNet18 與 EfficientNet，並利用晶圓圖具有的「廉價旋轉/翻轉空間對稱性」進行數據增強（Data Augmentation），以提升模型對於各種缺陷模式（如 Scratch, Ring, Cluster）的辨識率。


### 1. 晶圓圖特有的數據增強：旋轉與翻轉 (Data Augmentation)

- 實作：晶圓圖具有高度的空間對稱性（圓形結構）。一個特定的刮痕（Scratch）或邊緣環狀缺陷（Edge Ring），無論在晶圓上旋轉 90 度、180 度或進行水平/垂直翻轉，其缺陷的物理成因和類型完全不會改變。這種「數據增強」在半導體影像領域中成本極低（不需人工重新標註），防禦模型過擬合的效果卻極佳。


In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

# 定義晶圓圖專用的資料增強轉換 Pipeline
# 由於晶圓圖是 2D 矩陣轉成的張量，我們主要使用幾何轉換
wafer_transform = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.RandomRotation(degrees=(0, 360)),  # 360度隨機旋轉
        transforms.RandomHorizontalFlip(p=0.5),  # 50% 機率水平翻轉（跳躍對稱）
        transforms.RandomVerticalFlip(p=0.5),  # 50% 機率垂直翻轉
        transforms.ToTensor(),  # 轉回 PyTorch 張量
    ]
)


# 自訂晶圓圖資料集 (WaferDataset)
class WaferMapDataset(Dataset):
    def __init__(self, num_samples=100, transform=None):
        self.transform = transform
        # 模擬 100 片 224x224 單通道 (灰階) 的晶圓缺陷圖
        self.images = np.random.randint(
            0, 5, size=(num_samples, 224, 224), dtype=np.uint8
        )
        # 模擬缺陷標籤 (0: Scratch, 1: Ring, 2: Cluster)
        self.labels = np.random.randint(0, 3, size=(num_samples,))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)
        else:
            img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)

        return img, label


# 驗證資料載入器
dataset = WaferMapDataset(num_samples=10, transform=wafer_transform)
loader = DataLoader(dataset, batch_size=2, shuffle=True)
images, labels = next(iter(loader))
print(f"資料載入成功。Batch 影像形狀: {images.shape} | 標籤形狀: {labels.shape}")

### 2. 骨幹網路微調：WaferResNetClassifier 與 EfficientNet 設計

- 實作：在業界專案中，我們很少從頭盲目訓練一個全新 CNN，而是基於預訓練（Pre-trained）的 ResNet18 或 EfficientNet 進行微調。由於晶圓圖原本是單通道或特定的晶粒格式，我們需要改寫模型的第一層卷積核（Input Channel 改為 1）與最後一層全連接層（Output Classes 改為 3）。


In [ ]:
import torchvision.models as models
import torch.nn as nn


class WaferResNetClassifier(nn.Module):
    """對應專案：WaferResNetClassifier 分類器設計"""

    def __init__(self, num_classes=3):
        super(WaferResNetClassifier, self).__init__()
        # 載入標準 ResNet18（可選擇預訓練權重，此處演示基礎骨幹結構）
        self.resnet = models.resnet18(weights=None)

        # 關鍵改寫 A：ResNet 預設輸入是 RGB 3通道，我們修改第一層使其能接收 1 通道的晶圓圖
        self.resnet.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False,
        )

        # 關鍵改寫 B：將最後的輸出全連接層 (Linear Layer) 的輸出維度，改為我們專案的缺陷類別數
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)


class WaferEfficientNetClassifier(nn.Module):
    """對應專案對照組：EfficientNet-B0 分類器設計"""

    def __init__(self, num_classes=3):
        super(WaferEfficientNetClassifier, self).__init__()
        self.efficientnet = models.efficientnet_b0(weights=None)

        # 改寫第一層卷積
        old_conv = self.efficientnet.features[0][0]
        self.efficientnet.features[0][0] = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )

        # 改寫最後一層分類頭 (Classifier Head)
        in_features = self.efficientnet.classifier[1].in_features
        self.efficientnet.classifier[1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.efficientnet(x)


# 實體化測試
model_resnet = WaferResNetClassifier(num_classes=3)
mock_input = torch.randn(2, 1, 224, 224)  # 模擬 2 片晶圓影像
output = model_resnet(mock_input)
print(f"ResNet 輸出張量形狀: {output.shape} -> [Batch_Size, Num_Classes] 成功！")

### 3. 模型前向推理與缺陷分類流水線 (classify_defects)

- 實作：編寫核心的 classify_defects() 函數，將上述模型包裝成產線能直接呼叫的介面，輸入原始晶圓數據，輸出預測的缺陷類別與機率值。


In [ ]:
def classify_defects(wafer_tensor: torch.Tensor, model: nn.Module) -> dict:
    """對應專案：classify_defects() 前向推理封裝"""
    model.eval()  # 切換至評估模式

    # 確保張量維度正確 [1, 1, H, W]
    if len(wafer_tensor.shape) == 3:
        wafer_tensor = wafer_tensor.unsqueeze(0)

    with torch.no_grad():
        logits = model(wafer_tensor)
        # 透過 Softmax 將點數轉換為機率分佈
        probabilities = torch.softmax(logits, dim=1).numpy()[0]
        predicted_class_idx = np.argmax(probabilities)

    classes_mapping = {
        0: "Scratch (刮痕缺陷)",
        1: "Edge Ring (邊緣環狀缺陷)",
        2: "Cluster (群聚型缺陷)",
    }

    return {
        "predicted_defect": classes_mapping[predicted_class_idx],
        "confidence": float(probabilities[predicted_class_idx]),
        "all_probabilities": {
            classes_mapping[i]: float(probabilities[i]) for i in range(3)
        },
    }


# ---- 模擬測試推理流水線 ----
single_wafer = torch.randn(1, 224, 224)  # 模擬單片待測晶圓
inference_result = classify_defects(single_wafer, model_resnet)

print("\n 測試單片晶圓影像分類結果:")
print(f"判定缺陷類型: {inference_result['predicted_defect']}")
print(f"模型信心水準: {inference_result['confidence'] * 100:.2f}%")

- 總結：在我們處理晶圓缺陷分類（Wafer Bin Map Classification）時，電腦視覺的引入能大幅取代過去人工目檢的負擔。在我的專案中，我獨立設計了 WaferResNetClassifier 架構。我利用 Pytorch 修改了經典 ResNet18 與 EfficientNet 的底層結構，將輸入通道改為符合晶圓圖的單通道灰階格式，並改寫最終的 Dense Head 以精準預測 Scratch 或 Cluster 等特定缺陷。此外，考量到半導體樣本珍貴，我充分利用了晶圓圖物理上的對稱特性，在 Pipeline 最上游加入了 RandomRotation (360度隨機旋轉) 與 跳躍翻轉 的數據增強策略。這等同於免費獲得了數倍的訓練樣本，能有效摧毀神經網路的過擬合現象，大幅提高模型在實際產線跨批次測試時的泛化能力與信心度（Confidence Score）。
